# Chapter 53: Audio-Visual Learning & Fusion
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Memahami arsitektur integrasi video dan sinyal akustik berpusat pada visi komputer (*vision-centric*), menghitung korelasi silang temporal untuk mendeteksi jeda sinkronisasi $\tau^*$, menerapkan fusi terbobot keandalan untuk verifikasi deteksi peristiwa, serta mengevaluasi kemampuan penolakan alarm palsu visual.

### 1. Landasan Teori & Formulasi Sinkronisasi Audio-Visual
Korelasi silang diskrit pergeseran waktu lag $\tau$:
$$R_{va}(\tau) = \sum_{t} E_v(t) E_a(t + \tau), \quad \tau^* = \arg\max_{\tau} R_{va}(\tau)$$

Fusi Keputusan Terbobot Keandalan:
$$P_{\text{AV}} = \frac{\gamma_v P_v + \gamma_a P_a}{\gamma_v + \gamma_a}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Korelasi Silang Temporal & Validasi Seksi 3 Part A
def cross_correlation_sync(ev, ea, max_lag=2):
    T = len(ev)
    corrs = {}
    for tau in range(-max_lag, max_lag + 1):
        tot = 0.0
        for t in range(T):
            ts = t + tau
            if 0 <= ts < T:
                tot += ev[t] * ea[ts]
        corrs[tau] = round(tot, 4)
    opt_lag = max(corrs, key=corrs.get)
    return corrs, opt_lag

# Data 5 Frame sesuai Seksi 3
E_v = np.array([0.10, 0.20, 0.90, 0.30, 0.10])
E_a = np.array([0.00, 0.10, 0.20, 0.90, 0.30])

correlations, optimal_tau = cross_correlation_sync(E_v, E_a, max_lag=2)
print("Hasil Korelasi Silang R(tau):", correlations)
print(f"Optimal Lag tau*: {optimal_tau} (Audio tertunda {optimal_tau} frame)")

# Assertions
assert correlations[0] == 0.5000
assert correlations[1] == 0.9500
assert correlations[-1] == 0.2400
assert optimal_tau == 1
print("VALIDASI BERHASIL: Nilai korelasi silang dan lag optimal cocok persis sesuai derivasi manual!")

In [ ]:
# 4. Eksperimen Verifikasi Peristiwa Keselamatan & Supresi False Alarm
def fuse_av(p_v, g_v, p_a, g_a):
    w_v = g_v / (g_v + g_a)
    w_a = g_a / (g_v + g_a)
    p_fused = w_v * p_v + w_a * p_a
    return p_fused, p_fused >= 0.50

# Kasus 1: Peristiwa Nyata (Tabrakan Malam Hari)
p_av1, conf1 = fuse_av(0.75, 0.70, 0.90, 0.85)
print(f"Kasus 1 (Tabrakan Riil): P_AV = {p_av1:.4f} -> Terverifikasi: {conf1}")
np.testing.assert_allclose(p_av1, 0.832258, atol=1e-4)
assert conf1 is True

# Kasus 2: Alarm Palsu Visual (Bayangan Melintas Cepat, Audio Hening)
p_av2, conf2 = fuse_av(0.75, 0.70, 0.05, 0.85)
print(f"Kasus 2 (Alarm Palsu): P_AV = {p_av2:.4f} -> Terverifikasi: {conf2}")
np.testing.assert_allclose(p_av2, 0.366129, atol=1e-4)
assert conf2 is False, "Alarm palsu harus tertolak!"
print("VERIFIKASI SISTEM SELESAI: Alarm palsu visual berhasil ditolak secara deterministik.")

In [ ]:
# 5. Spesifikasi Pipeline PyTorch / Audio Spectrogram (Untuk GPU / Colab)
if HAS_TORCH:
    print("Mode PyTorch / GPU Aktif.")
    v_tensor = torch.tensor(E_v, dtype=torch.float32).unsqueeze(0)
    a_tensor = torch.tensor(E_a, dtype=torch.float32).unsqueeze(0)
    print("Shape Visual Tensor:", v_tensor.shape)
else:
    print("PANDUAN EKSEKUSI MODEL AUDIO-VISUAL DI GOOGLE COLAB:")
    print("1. Pasang paket torchaudio dan torchvision:")
    print("   !pip install torchaudio torchvision librosa transformers")
    print("2. Pipeline Ekstraksi Fitur Spektrogram:")
    print("   import torchaudio.transforms as T")
    print("   mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, n_mels=80)")
    print("   log_mel = torch.log(mel_transform(audio_waveform) + 1e-6)")
    print("3. Arsitektur Unggulan:")
    print("   - Multimodal Bottleneck Transformer (MBT)")
    print("   - AV-Hubert (Audio-Visual Speech Recognition)")
    print("   - Visually-Guided Audio Source Separation")

### 6. Rangkuman & Pembahasan Akademik
1. **Penyelarasan Waktu**: Korelasi silang energi temporal adalah alat matematis mendasar untuk mendeteksi dan mengompensasi jeda sinkronisasi antar-sensor.
2. **Vision-Centric Synergy**: Pengamatan visual memberikan lokalisasi spasial adegan, sementara bukti akustik memperkuat atau menolak hipotesis peristiwa.
3. **Supresi Alarm Palsu**: Fusi keandalan terbukti efektif menolak alarm palsu yang timbul dari artefak pencahayaan visual.